# Custom Climate Profiles Generation

<br> Many energy system modeling and other planning processes are designed to intake hourly information in order to capture short term fluctuations in conditions that affect energy supply and demand. An *Annualized Hourly Climate Profile* is a dataset that contains hourly weather conditions at a given location for an entire year, and is commonly referred to as an **8760 Climate Profile**. Climate profiles can represent future weather conditions in a changing climate and can thus better inform design and planning processes for a wide range of future needs. Guidance, methodologies, and recommendations for <span style="color: blue;\">[Cal-Adapt Climate Profiles](https://analytics.cal-adapt.org/climate-profiles/)</span>  is available for reference. A comprehensive set of pre-generated Climate Profiles for the most common use-cases is available via the <span style="color: blue;\">[Cal-Adapt Data Catalog](https://analytics.cal-adapt.org/climate-profiles/accessing_the_data/)</span>, and an easy-to-use data download API tool is coming soon!

On the Cal-Adapt: Analytics Engine, we currently provide two kinds of Climate Profiles: 
1. **Standard Year 8760**:  One year of hourly data that represents any desired statistical percentile of weather conditions for a location over a 30-year climatological period. A Standard Year builds a climate profile for each climate model on a single variable and can be used to evaluate both median and extreme conditions by selecting appropriate percentiles. Both historical and future Standard Year 8760s are available.

2. **Typical Meteorological Year 8760 (TMY)**: One year of hourly data that represents the median weather conditions for a location over a climatological period. A TMY is built from ten specific weather variables that are weighted in compliance with <span style="color: blue;\">[TMY standard methodology](https://nsrdb.nrel.gov/data-sets/tmy)</span>. TMYs statistically assess the median conditions from 30 years of model data and select the most “typical” month for each month during a year. These are then compiled into a single climate profile. The resulting TMY file includes multiple variables and has very specific formatting requirements. Both historical and future TMY (FTMY) 8760s are available.

**Intended Application**: As a user, I want to <span style="color:#FF0000">**generate a custom climate profile**</span> for my area:
1. Standard Year 8760 profile, with my variable and extreme conditions of choice.
2. TMY 8760 profile, for the closest weather station to my area. 

#### Step 0: Set-Up
Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [ ]:
from climakitae.explore.standard_year_profile import (
    get_climate_profile,
    export_profile_to_csv,
)
from climakitae.explore.typical_meteorological_year import TMY
from climakitae.core.data_interface import get_subsetting_options

import warnings

warnings.filterwarnings("ignore")

### Generate a Custom Standard Year Profile 

AE makes it easy to generate a custom Standard Year climate profile for you. Options for customization include: 

|**Argument**|**Options**|**Notes**|
|------------|-----------|-----------------------|
|variable|WRF variables and units| Check out our variable and units list <span style="color: blue;\">[here](https://github.com/cal-adapt/climakitae/blob/main/climakitae/data/variable_descriptions.csv)</span>.|
|units| WRF variables and units|Check out our variable and units list <span style="color: blue;\">[here](https://github.com/cal-adapt/climakitae/blob/main/climakitae/data/variable_descriptions.csv)</span>.|
|qtile|0-1|Statistical quantile to generate Standard year profile; median = 0.5|
|no_delta|True, False |Option to take a delta signal difference between your selected GWL and the historical reference GWL 1.2°C. Default = False|
|resolution|"3 km", "9 km", "45 km"| Spatial resolution of input data. Default = "3 km"|
|Location: stations|HadISD weather stations | Check out the <span style="color: blue;\">[available station list](https://github.com/cal-adapt/climakitae/blob/main/climakitae/data/hadisd_stations.csv)</span> for station options. Cached areas include counties, demand forecast zones, or service territories.|
|Location: latitude, longitude|Custom lat-lon pairs| Note: there is a small buffer applied to lat-lon arguments to retrieve the closest gridcell. |
|Location: cached_area|Counties, demand forecast zones, service territories | For the full list of cached areas available on AE, the `get_subsetting_options` function can help. Check out the <span style="color: blue;\">[basic_data_access notebook](https://github.com/cal-adapt/cae-notebooks/blob/main/data-access/basic_data_access.ipynb)</span> for how to use this function. |
|approach|"Warming Level", "Time"|Default = "Warming Level"|
|warming_level|Valid range is 0.5 - 4.0°C|Fully customizeable GWL. Default GWL = 1.2°C |
|warming_level_window|Valid range is 5-25 years|Warming level window size. Default = 15 years (30-year window)|
|centered_year|Valid range is 1980 - 2099| Required for time-based approach. |
|time_profile_scenario|"SSP2-4.5", "SSP3-7.0", "SSP5-8.5"| Option to return a specified SSP scenario for time-based approach. Default="SSP3.7-0". Note: "SSP 2-4.5", "SSP 5-8.5" only valid if spatial resolution is "9 km", "45 km" and does not include bias-adjusted models |
|bias_adjusted_models| True, False| Option to return only bias-adjusted models. Default=False|

Now, set-up the profile generator -- the `get_climate_profile` function does all the work for you. Modify any argument based on your desired custom profile, options for each argument are listed in the table above!

In [ ]:
# Set up the Standard Year generator
profile_selections = {  
    ## Required variable and profile arguments
    "variable": "Air Temperature at 2m",
    "resolution": "3 km",
    "q": 0.5,
    "units": "degF",

    ## Required approach arguments, Options: "Warming Level", "Time"
    "approach": "Warming Level",
    # "warming_level": [warming_level], # GWL-option only
    # "centered_year": centered_year, # Time-based option only
    
    ## Required location arguments -- Uncomment your desired location option and modify
    "stations": ["Sacramento Executive Airport (KSAC)"], 
    # "latitude": (latitude - 0.02, latitude + 0.02), 
    # "longitude": (longitude - 0.02, longitude + 0.02),  
    # "cached_area": area_name, 

    ## Additional optional arguments -- Uncomment any desired options and modify
    # "no_delta": True, 
    # "warming_level_window": 10,
    # "time_profile_scenario": "SSP2-4.5,
    # "bias_adjusted_models": True,
}

# Generate the climate profile
profile = get_climate_profile(**profile_selections)

**Export**. The following cell will export your custom generated Standard Year profile!

In [ ]:
# the function uses the previously defined profile selections to generate the output file name
export_profile_to_csv(profile, **profile_selections)

### Generate a Typical Meteorological Year (TMY) Profile

AE makes it easy to generate a custom Typical Meteorological Year climate profile for you. You can generate both a historical TMY and a future TMY (FTMY). Options for customization include: 


|**Argument**|**Options**|**Notes**|
|------------|-----------|-----------------------|
|Location: stations|HadISD weather stations | Check out the <span style="color: blue;\">[available station list](https://github.com/cal-adapt/climakitae/blob/main/climakitae/data/hadisd_stations.csv)</span> for station options. Cached areas include counties, demand forecast zones, or service territories.|
|Location: latitude, longitude|Custom lat-lon pairs| Note: there is a small buffer applied to lat-lon arguments to retrieve the closest gridcell. |
|approach|"Warming Level", "Time"|Default = "Warming Level"|
|warming_level|Valid range is 0.5 - 4.0°C|Fully customizeable GWL. Default GWL = 1.2°C |
|start_year|Valid range is 1980-2099. | We recommend a 30-year period such as 1990-2020. |
|end_year|Valid range is 1980-2099.  | We recommend a 30-year period such as 1990-2020. |

Note: The TMY methodology here mirrors that of the NSRDB TMY3 methodology which heavily weights the solar radiation input data. Be aware that the final selection of "typical" months may not be typical for other variables. Because a TMY represents average rather than extreme conditions, an TMY profile is not suited for designing systems to meet the worst-case conditions occurring at a location. Extreme TMY profiles (XMY) are coming to AE soon. 

In [ ]:
# Set up the TMY profile generator! The verbose option will output progress of the TMY generation
tmy = TMY(
    ## Location -- uncomment your desired option
    station_name = "Sacramento Executive Airport (KSAC)",
    # latitude = latitude,
    # longitude = longitude,
    
    # Approach -- uncomment your desired option
    warming_level = warming_level,
    # start_year = start_year,
    # end_year = end_year,
    verbose=True,
)

# Generate the profile!
tmy.generate_tmy()

**Export to non-EPW format.** TMY profiles are exported in .epw format by default, but can be exported as both `.csv` and `.tmy` file formats using the method `export_tmy_data` with the argument `extension="csv"` as shown below.

In [ ]:
tmy.export_tmy_data(extension="csv")